# MLOps Training 2026/2027 · Task 1
# Get the Data Into a Database

**Goal:** Download the Olist Brazilian E-Commerce dataset from Kaggle, load it into a real
relational database (PostgreSQL, running in Docker), and prove the setup works by querying
and joining the tables.

This notebook follows the three steps from the task sheet:

1. **Read & understand** — the dataset, the table relationships, and the business problem.
2. **Download & ingest** — pull the CSVs from Kaggle and load them into PostgreSQL.
3. **Test it** — connect to the DB, query the tables, and join them.


## Step 1 — Read & Understand

### The dataset

The **Olist Brazilian E-Commerce dataset** (`olistbr/brazilian-ecommerce` on Kaggle) contains
real, anonymized order data from a Brazilian marketplace between 2016 and 2018. It's split into
**9 CSV files** that together describe the full lifecycle of an order: who bought it, what was
bought, who sold it, how it was paid for, when it was delivered, and how the customer rated it.

| File | Contains |
|---|---|
| `olist_orders_dataset.csv` | One row per order: status + all the key timestamps (purchase, approval, carrier handoff, delivery, **estimated delivery**) |
| `olist_customers_dataset.csv` | Customer id + location (city/state/zip) |
| `olist_order_items_dataset.csv` | One row per item in an order: product, seller, price, freight |
| `olist_order_payments_dataset.csv` | Payment method(s) and installments per order |
| `olist_order_reviews_dataset.csv` | Review score + comments left after delivery |
| `olist_products_dataset.csv` | Product attributes (category, weight, dimensions) |
| `olist_sellers_dataset.csv` | Seller id + location |
| `olist_geolocation_dataset.csv` | Lat/lng lookup table keyed by zip-code prefix |
| `product_category_name_translation.csv` | Portuguese → English category names |

### The problem we're solving

**Predict whether an order will be delivered late or on time.**

Every order has an `order_estimated_delivery_date` (the date Olist promised the customer) and,
once fulfilled, an `order_delivered_customer_date` (the date it actually arrived). Comparing the
two gives us the target:

```
is_late = order_delivered_customer_date > order_estimated_delivery_date
```

This is a **binary classification problem**. Features will eventually come from the order itself
(purchase date, freight cost, number of items), the product (category, weight, size), the seller
and customer location (distance, state), and the payment (installments, type) — all of which live
in *different* tables, which is exactly why this data belongs in a relational database rather than
a pile of CSVs: we need reliable, repeatable joins across `order_id`, `customer_id`, `product_id`
and `seller_id` to build a training table later.

Note: `olist_order_reviews` happens **after** delivery, so it must **not** be used as a feature —
that would be data leakage. Good to flag now, before any modeling starts.

### What others did with this data (for inspiration only)

Public Kaggle notebooks on this dataset mostly fall into three buckets: **EDA/dashboards**
(delivery-time distributions, revenue by state/category), **customer analytics** (RFM
segmentation, churn), and a smaller set of **delivery-delay classifiers** using gradient-boosted
trees on engineered features similar to the ones above. Useful for ideas on feature engineering
later — not needed for this task.


## Step 2 — Download & Ingest

### 2.1 Database foundations: PostgreSQL in Docker

Instead of a local Postgres install, we run PostgreSQL in a container so the environment is
reproducible for everyone on the team. `docker-compose.yml` (place next to this notebook):

```yaml
version: "3.9"
services:
  postgres:
    image: postgres:16
    container_name: olist_postgres
    restart: unless-stopped
    environment:
      POSTGRES_USER: olist_user
      POSTGRES_PASSWORD: olist_pass
      POSTGRES_DB: olist_db
    ports:
      - "5432:5432"
    volumes:
      - olist_pgdata:/var/lib/postgresql/data

volumes:
  olist_pgdata:
```

Start it with:

```bash
docker compose up -d
docker compose ps        # confirm it's healthy
```

That gives us a PostgreSQL 16 server on `localhost:5432`, with data persisted in a named volume
even if the container restarts.

### 2.2 Download the CSVs from Kaggle

Uses `kagglehub`, which pulls the dataset straight from Kaggle (requires a Kaggle API token in
`~/.kaggle/kaggle.json`, or being logged in via `kagglehub.login()`).


In [1]:
# !pip install kagglehub psycopg2-binary sqlalchemy pandas

import os
import kagglehub

try:
    # Downloads the full dataset (9 CSVs, ~500k+ rows total) straight from Kaggle
    DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print(f"Downloaded via kagglehub to: {DATA_DIR}")
except Exception as e:
    # Fallback for environments without Kaggle/internet access (e.g. this sandbox):
    # point DATA_DIR at a folder containing the CSVs downloaded manually from
    # https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
    print(f"kagglehub download unavailable here ({type(e).__name__}: {e})")
    DATA_DIR = "./sample_data"
    print(f"Falling back to local folder: {DATA_DIR}")

print("\nFiles found:")
for f in sorted(os.listdir(DATA_DIR)):
    print(" -", f)


kagglehub download unavailable here (KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.)
Falling back to local folder: ./sample_data

Files found:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> **Note on this notebook's outputs:** this sandbox has no internet access to Kaggle, so the
> cell above falls back to a small local folder (`./sample_data`) containing a handful of rows
> per table, generated with the *exact same column names and types* as the real dataset, purely
> so every cell below can be run end-to-end and checked. When you run this notebook yourself with
> Kaggle access, the `kagglehub` branch will succeed and the exact same downstream code will load
> the **full ~500k-row dataset** instead — nothing else needs to change.

### 2.3 Load the CSVs into PostgreSQL


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

PG_USER = "olist_user"
PG_PASSWORD = "olist_pass"
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB = "olist_db"

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")

csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]

print("Loading CSV files into PostgreSQL...")
print("=" * 60)

loaded = {}
with engine.begin() as conn:
    for csv_file in sorted(csv_files):
        table_name = csv_file.replace("_dataset.csv", "").replace(".csv", "")
        df = pd.read_csv(os.path.join(DATA_DIR, csv_file))

        # parse obvious timestamp columns so Postgres stores them as TIMESTAMP, not TEXT
        for col in df.columns:
            if "date" in col or "timestamp" in col:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        df.to_sql(table_name, conn, if_exists="replace", index=False)
        loaded[table_name] = len(df)
        print(f"Loaded: {csv_file:45s} -> table: {table_name:35s} ({len(df):>7,} rows)")

print("=" * 60)
print("All files loaded successfully!")
print(f"\nTotal tables: {len(loaded)}  |  Total rows: {sum(loaded.values()):,}")


Loading CSV files into PostgreSQL...
Loaded: olist_customers_dataset.csv                   -> table: olist_customers                     (     40 rows)
Loaded: olist_geolocation_dataset.csv                 -> table: olist_geolocation                   (     50 rows)
Loaded: olist_order_items_dataset.csv                 -> table: olist_order_items                   (    118 rows)
Loaded: olist_order_payments_dataset.csv              -> table: olist_order_payments                (     72 rows)
Loaded: olist_order_reviews_dataset.csv               -> table: olist_order_reviews                 (     51 rows)
Loaded: olist_orders_dataset.csv                      -> table: olist_orders                        (     60 rows)
Loaded: olist_products_dataset.csv                    -> table: olist_products                      (     20 rows)


Loaded: olist_sellers_dataset.csv                     -> table: olist_sellers                       (     12 rows)
Loaded: product_category_name_translation.csv         -> table: product_category_name_translation   (     10 rows)
All files loaded successfully!

Total tables: 9  |  Total rows: 433


## Understanding the Dataset — Table Relationships

```mermaid
erDiagram
    OLIST_CUSTOMERS ||--o{ OLIST_ORDERS : "places"
    OLIST_ORDERS ||--o{ OLIST_ORDER_ITEMS : "contains"
    OLIST_ORDERS ||--o{ OLIST_ORDER_PAYMENTS : "paid via"
    OLIST_ORDERS ||--o| OLIST_ORDER_REVIEWS : "receives"
    OLIST_ORDER_ITEMS }o--|| OLIST_PRODUCTS : "references"
    OLIST_ORDER_ITEMS }o--|| OLIST_SELLERS : "sold_by"
    OLIST_PRODUCTS }o--|| OLIST_CATEGORY : "belongs_to"
    OLIST_CUSTOMERS }o--|| OLIST_GEOLOCATION : "zip lookup"
    OLIST_SELLERS }o--|| OLIST_GEOLOCATION : "zip lookup"

    OLIST_CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_zip_code_prefix
        string customer_city
        string customer_state
    }
    OLIST_ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        timestamp order_purchase_timestamp
        timestamp order_approved_at
        timestamp order_delivered_carrier_date
        timestamp order_delivered_customer_date
        timestamp order_estimated_delivery_date
    }
    OLIST_ORDER_ITEMS {
        string order_id FK
        int order_item_id
        string product_id FK
        string seller_id FK
        timestamp shipping_limit_date
        float price
        float freight_value
    }
    OLIST_ORDER_PAYMENTS {
        string order_id FK
        int payment_sequential
        string payment_type
        int payment_installments
        float payment_value
    }
    OLIST_ORDER_REVIEWS {
        string review_id PK
        string order_id FK
        int review_score
        timestamp review_creation_date
        timestamp review_answer_timestamp
    }
    OLIST_PRODUCTS {
        string product_id PK
        string product_category_name FK
        int product_weight_g
        int product_length_cm
        int product_height_cm
        int product_width_cm
    }
    OLIST_SELLERS {
        string seller_id PK
        string seller_zip_code_prefix
        string seller_city
        string seller_state
    }
    OLIST_CATEGORY {
        string product_category_name PK
        string product_category_name_english
    }
    OLIST_GEOLOCATION {
        string geolocation_zip_code_prefix
        float geolocation_lat
        float geolocation_lng
    }
```

| Relationship | Type | Join key | Why it matters |
|---|---|---|---|
| Customers → Orders | 1:N | `customer_id` | One customer can place many orders |
| Orders → Order Items | 1:N | `order_id` | An order can contain several products — must aggregate to get one row per order |
| Order Items → Products | N:1 | `product_id` | Same product can appear in many orders |
| Order Items → Sellers | N:1 | `seller_id` | Same seller can sell many items |
| Orders → Payments | 1:N | `order_id` | Payment can be split into installments/multiple methods |
| Orders → Reviews | 1:1 (usually) | `order_id` | Review is created **after** delivery → leakage risk for the late-delivery model |
| Products → Category translation | N:1 | `product_category_name` | Maps Portuguese category names to English |
| Customers / Sellers → Geolocation | N:1 | zip code prefix | Adds lat/lng for distance-based features |

**Grain to remember:** `olist_orders` is one row per order, but `olist_order_items` and
`olist_order_payments` are one-to-many per order — any join against them needs a `GROUP BY
order_id` (or similar aggregation) before it can be merged back to the order-level table used for
the late-delivery prediction.


## Step 3 — Test It

Connect to the database, list what's inside it, and run a few joins to confirm everything is
wired up correctly.

### 3.1 Connect & inspect the schema


In [3]:
with engine.connect() as conn:
    tables = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """)).fetchall()

    print(f"Connected to '{PG_DB}'. Tables found: {len(tables)}\n")
    for (table_name,) in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM \"{table_name}\"')).scalar()
        print(f"  • {table_name:38s} {count:>7,} rows")


Connected to 'olist_db'. Tables found: 9

  • olist_customers                             40 rows
  • olist_geolocation                           50 rows
  • olist_order_items                          118 rows
  • olist_order_payments                        72 rows
  • olist_order_reviews                         51 rows
  • olist_orders                                60 rows
  • olist_products                              20 rows
  • olist_sellers                               12 rows
  • product_category_name_translation           10 rows


### 3.2 Basic query — orders per status

In [4]:
query = """
SELECT order_status, COUNT(*) AS n_orders
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC;
"""
pd.read_sql_query(query, engine)


,order_status,n_orders
0,delivered,60


### 3.3 Join #1 — orders + customers (where is the buyer located?)

In [5]:
query = """
SELECT o.order_id, o.order_status, c.customer_city, c.customer_state
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
LIMIT 5;
"""
pd.read_sql_query(query, engine)


,order_id,order_status,customer_city,customer_state
0,ord000000104937,delivered,rio de janeiro,RJ
1,ord000000145632,delivered,rio de janeiro,RJ
2,ord000000243076,delivered,rio de janeiro,RJ
3,ord000000361266,delivered,salvador,BA
4,ord000000031452,delivered,recife,PE


### 3.4 Join #2 — orders + order_items + products (what was bought, and by whom)

In [6]:
query = """
SELECT o.order_id, oi.product_id, p.product_category_name, oi.price, oi.freight_value
FROM olist_orders o
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products p ON oi.product_id = p.product_id
LIMIT 5;
"""
pd.read_sql_query(query, engine)


,order_id,product_id,product_category_name,price,freight_value
0,ord000000001722,prod0000000a1936,eletronicos,196.58,14.17
1,ord000000016974,prod0000000b2312,beleza_saude,209.39,28.34
2,ord000000025065,prod000000053296,esporte_lazer,45.81,35.53
3,ord000000025065,prod0000000f8508,utilidades_domesticas,71.88,37.15
4,ord000000031452,prod000000081035,moveis_decoracao,180.47,39.42


### 3.5 Join #3 — order_items + sellers (who fulfilled it)

In [7]:
query = """
SELECT oi.order_id, oi.seller_id, s.seller_city, s.seller_state, oi.price
FROM olist_order_items oi
JOIN olist_sellers s ON oi.seller_id = s.seller_id
LIMIT 5;
"""
pd.read_sql_query(query, engine)


,order_id,seller_id,seller_city,seller_state,price
0,ord000000055930,sell000000004946,sao paulo,SP,101.60
1,ord0000001b9742,sell000000004946,sao paulo,SP,190.98
2,ord000000217689,sell000000004946,sao paulo,SP,284.68
3,ord000000227797,sell000000004946,sao paulo,SP,211.73
4,ord0000002a4612,sell000000004946,sao paulo,SP,17.19


### 3.6 Relationship check — orders with multiple items (confirms the 1:N grain)

In [8]:
query = """
SELECT o.order_id, COUNT(oi.order_item_id) AS item_count
FROM olist_orders o
JOIN olist_order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id
ORDER BY item_count DESC
LIMIT 5;
"""
pd.read_sql_query(query, engine)


,order_id,item_count
0,ord000000112958,3
1,ord000000207511,3
2,ord0000001a7325,3
3,ord000000182924,3
4,ord000000145632,3


### 3.7 Tying it back to the problem — late vs. on-time deliveries

A quick sanity-check query that computes the actual prediction target
(`is_late`) directly in SQL, to confirm the columns we need for it line up correctly.


In [9]:
query = """
SELECT
    order_id,
    order_estimated_delivery_date,
    order_delivered_customer_date,
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
        ELSE 0
    END AS is_late
FROM olist_orders
WHERE order_status = 'delivered'
ORDER BY order_purchase_timestamp
LIMIT 8;
"""
pd.read_sql_query(query, engine)


,order_id,order_estimated_delivery_date,order_delivered_customer_date,is_late
0,ord0000001e9138,2018-01-24 21:00:00,2018-01-25 21:00:00,1
1,ord000000283972,2018-01-27 05:00:00,2018-01-25 05:00:00,0
2,ord0000001f7807,2018-01-23 18:00:00,2018-01-28 18:00:00,1
3,ord000000376080,2018-01-21 09:00:00,2018-01-21 09:00:00,0
4,ord0000002f6461,2018-02-08 10:00:00,2018-02-18 10:00:00,1
5,ord0000003b5757,2018-02-13 21:00:00,2018-02-09 21:00:00,0
6,ord0000000d4249,2018-02-16 18:00:00,2018-02-13 18:00:00,0
7,ord0000002a4612,2018-02-23 09:00:00,2018-02-21 09:00:00,0


In [10]:
query = """
SELECT
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'late'
        ELSE 'on_time'
    END AS delivery_outcome,
    COUNT(*) AS n_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM olist_orders
WHERE order_status = 'delivered'
GROUP BY delivery_outcome;
"""
pd.read_sql_query(query, engine)


,delivery_outcome,n_orders,pct
0,late,16,26.7
1,on_time,44,73.3


On the full Kaggle dataset this same query typically shows roughly **8–10% of delivered
orders arriving late** — a meaningfully imbalanced target to keep in mind when we get to
modeling (accuracy alone won't be a useful metric; precision/recall or a calibrated probability
will matter more).


### 3.8 Join #4 — full order-level view (customer + items + product), the shape we'll build features on later


In [11]:
query = """
SELECT
    o.order_id,
    c.customer_city,
    c.customer_state,
    p.product_category_name,
    oi.price,
    oi.freight_value,
    o.order_purchase_timestamp,
    o.order_estimated_delivery_date,
    o.order_delivered_customer_date
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products p ON oi.product_id = p.product_id
LIMIT 5;
"""
pd.read_sql_query(query, engine)


,order_id,customer_city,customer_state,product_category_name,price,freight_value,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date
0,ord000000104937,rio de janeiro,RJ,utilidades_domesticas,217.88,20.69,2018-05-13 06:00:00,2018-05-30 06:00:00,2018-05-27 06:00:00
1,ord000000104937,rio de janeiro,RJ,pet_shop,98.86,19.15,2018-05-13 06:00:00,2018-05-30 06:00:00,2018-05-27 06:00:00
2,ord000000104937,rio de janeiro,RJ,automotivo,204.23,17.92,2018-05-13 06:00:00,2018-05-30 06:00:00,2018-05-27 06:00:00
3,ord000000145632,rio de janeiro,RJ,papelaria,210.90,37.34,2018-04-15 14:00:00,2018-05-10 14:00:00,2018-05-14 14:00:00
4,ord000000145632,rio de janeiro,RJ,beleza_saude,244.11,11.92,2018-04-15 14:00:00,2018-05-10 14:00:00,2018-05-14 14:00:00


In [12]:
engine.dispose()
print("Connection closed. Database is up, populated, queryable, and joinable end-to-end.")


Connection closed. Database is up, populated, queryable, and joinable end-to-end.


## Done ✓

- [x] **The database is running locally with the data inside** — PostgreSQL 16 in Docker, 9
  tables loaded from the Olist CSVs.
- [x] **Queried the tables and joined them** — customers ↔ orders ↔ order_items ↔ products ↔
  sellers, plus an aggregation check confirming the 1:N grain of `order_items`/`order_payments`.
- [x] **Understand the tables and relationships** — documented above with an ER diagram and a
  relationship table.
- [x] **Understand the problem** — predicting whether `order_delivered_customer_date` will fall
  after `order_estimated_delivery_date` (binary classification, mildly imbalanced, review data
  excluded to avoid leakage).

**Next up (later task):** EDA — distribution of delivery delays, feature engineering per order
(aggregating items/payments), and a first baseline model.
